# Lezione 5 — Waffle chart, Word Cloud e Seaborn

In questa lezione vediamo tre strumenti per dare impatto visivo e riassumere statistiche rapidamente:

1. **Waffle chart** con la libreria `pywaffle` (alternativa a un pie chart).
2. **Word cloud** con la libreria `wordcloud` (quali parole dominano un testo).
3. **Seaborn**: `countplot`, `barplot` e `regplot`, confrontati con lo scatter + `polyfit` manuale visto nelle lezioni precedenti.

> Nota: gli URL dei dati (`Canada.csv`, `alice_novel.txt`, `alice_mask.png`) sono gli stessi usati dal notebook ufficiale del corso (IBM Skills Network), quindi questo notebook funziona senza bisogno di file locali.

In [ ]:
# installare le due dipendenze extra che usiamo in questa lezione
!pip install pywaffle wordcloud

In [ ]:
# Import delle librerie
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from pywaffle import Waffle
from wordcloud import WordCloud, STOPWORDS
from PIL import Image
import urllib

mpl.style.use('ggplot')
%matplotlib inline

print('Librerie importate correttamente')

## 1. Caricare il dataset dell'immigrazione verso il Canada

Usiamo lo stesso dataset pulito (`Canada.csv`) del notebook ufficiale del corso, ospitato nel bucket pubblico di IBM Skills Network.

In [ ]:
df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')

# mettere 'Country' come indice, come nelle lezioni precedenti
df_can.set_index('Country', inplace=True)

print('Forma del dataset:', df_can.shape)
df_can.head()

## 2. Waffle chart

### L'idea

Un waffle chart è come un pie chart, ma in forma di griglia: si divide un rettangolo in celle (ad esempio 20x30 = 600 celle) e si colora ogni cella secondo la proporzione che spetta a ogni categoria.

### Il metodo manuale (solo per capire cosa succede dietro le quinte)

Costruirlo a mano implica: calcolare le proporzioni, definire la griglia, riempire una matrice NumPy, disegnarla con `plt.matshow()` e fabbricare una legenda a mano. È istruttivo, ma nessuno lo ripete ogni volta — per questo esiste `pywaffle`.

### Con `pywaffle` (il modo che userai nella pratica)

Ricreiamo il caso di Danimarca, Norvegia e Svezia.

In [ ]:
# sottoinsieme di paesi scandinavi
df_dsn = df_can.loc[['Denmark', 'Norway', 'Sweden'], :]
df_dsn[['Total']]

In [ ]:
# waffle chart con pywaffle: un'unica chiamata invece di ~40 righe manuali
fig = plt.figure(
    FigureClass=Waffle,
    rows=20,            # numero di righe della griglia
    columns=30,          # numero di colonne della griglia
    values=df_dsn['Total'],   # i dati: quanto occupa ogni categoria
    cmap_name='tab20',        # schema di colore
    legend={
        'labels': [f"{k} ({v})" for k, v in zip(df_dsn.index.values, df_dsn.Total)],
        'loc': 'lower left',
        'bbox_to_anchor': (0, -0.1),
        'ncol': 3
    }
)

plt.show()

Osserva la riga della legenda: la *list comprehension* costruisce automaticamente etichette come `"Sweden (5866)"` combinando l'indice del paese con il suo totale. Non serve scriverle a mano.

## 3. Word cloud

### L'idea

Una nuvola di parole mostra quali parole dominano un testo: quanto più una parola si ripete, più grande e in grassetto viene disegnata. Usiamo la libreria `wordcloud` (di Andreas Mueller) sul testo completo di *Alice's Adventures in Wonderland* (Lewis Carroll), scaricato dallo stesso URL pubblico usato dal notebook ufficiale del corso.

In [ ]:
# scaricare il testo completo del romanzo
alice_novel = urllib.request.urlopen('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/alice_novel.txt').read().decode("utf-8")

# stopword: parole vuote ("the", "and", ecc.) che non aggiungono informazioni
stopwords = set(STOPWORDS)

# istanziare l'oggetto WordCloud e generarlo a partire dal testo
alice_wc = WordCloud(background_color='white', stopwords=stopwords)
alice_wc.generate(alice_novel)

# WordCloud non si disegna da sola: ci serve matplotlib per mostrarla
plt.figure(figsize=(14, 18))
plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')   # una nuvola di parole non ha bisogno di assi
plt.show()

Se esegui questo senza filtrare nulla, parole come **"said"** dominano l'immagine senza fornire informazioni reali. La soluzione è aggiungerla manualmente a `stopwords` in base a ciò che vedi e rigenerare:

In [ ]:
stopwords.add('said')  # 'said' appare moltissimo ma non dice nulla di interessante
alice_wc.generate(alice_novel)  # rigenerare con il nuovo filtro

plt.figure(figsize=(14, 18))
plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')
plt.show()

### Nuvola di parole con maschera (mask)

`WordCloud` accetta un parametro `mask` con un'immagine affinché le parole si dispongano dentro quella forma invece che in un rettangolo. Usiamo la sagoma di Alice e del coniglio, anch'essa scaricata dall'URL pubblico del notebook ufficiale.

In [ ]:
# scaricare e convertire la maschera in un array NumPy
alice_mask = np.array(Image.open(urllib.request.urlopen('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%204/images/alice_mask.png')))

alice_wc = WordCloud(background_color='white', max_words=2000,
                      mask=alice_mask, stopwords=stopwords)
alice_wc.generate(alice_novel)

fig = plt.figure(figsize=(14, 18))
plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')
plt.show()

### Word cloud a partire da dati numerici

Possiamo anche generare una nuvola di parole a partire da dati che non sono testo narrativo: ripetiamo il nome di ogni paese di `df_can` un numero di volte proporzionale al suo peso nel totale dell'immigrazione, e passiamo quel testo generato a `WordCloud().generate()`.

In [ ]:
total_immigration = df_can['Total'].sum()

max_words = 90
word_string = ''
for country in df_can.index.values:
    # solo paesi con nome di una sola parola (semplifica l'esempio)
    if country.count(" ") == 0:
        repeat_num_times = int(df_can.loc[country, 'Total'] / total_immigration * max_words)
        word_string = word_string + ((country + ' ') * repeat_num_times)

wordcloud_countries = WordCloud(background_color='white').generate(word_string)

plt.figure(figsize=(14, 18))
plt.imshow(wordcloud_countries, interpolation='bilinear')
plt.axis('off')
plt.show()

## 4. Seaborn: statistica visuale

Seaborn è costruito sopra Matplotlib e conosce già le convenzioni della statistica visuale: conteggio per categoria, aggregazione con intervalli di confidenza, e adattamento + disegno di una regressione, tutto con meno codice di quanto serva in Matplotlib puro.

### `countplot`: conta righe per categoria

In [ ]:
df_can.reset_index(inplace=True)  # per avere 'Country' di nuovo come colonna, se dovesse servire

# quanti paesi sono registrati per continente?
plt.figure(figsize=(15, 10))
sns.countplot(x='Continent', data=df_can)
plt.show()

Se le etichette dell'asse x risultano troppo lunghe, possiamo ripulirle prima con `replace`:

In [ ]:
df_can1 = df_can.replace('Latin America and the Caribbean', 'L-America')
df_can1 = df_can1.replace('Northern America', 'N-America')

plt.figure(figsize=(15, 10))
sns.countplot(x='Continent', data=df_can1)
plt.show()

### `barplot`: aggrega un valore numerico per categoria

Mentre `countplot` conta *righe*, `barplot` aggrega un *valore numerico* (di default la media) raggruppando per una variabile categorica, e aggiunge automaticamente una barra di errore con l'intervallo di confidenza.

In [ ]:
plt.figure(figsize=(15, 10))
sns.barplot(x='Continent', y='Total', data=df_can1)
plt.show()

# possiamo verificare il calcolo noi stessi
df_can1.groupby('Continent')['Total'].mean()

### `regplot`: scatter + regressione in un'unica riga

Ecco il confronto che vale davvero la pena: nelle lezioni precedenti, per disegnare uno scatter con linea di tendenza usavamo `np.polyfit` + `np.poly1d` + `plt.scatter` + `plt.plot` (~20 righe). Con Seaborn è un'unica chiamata.

In [ ]:
years = list(map(str, range(1980, 2014)))

# totale di immigrazione per anno (tutte le righe sommate)
df_tot = pd.DataFrame(df_can[years].sum(axis=0))
df_tot.index = map(float, df_tot.index)
df_tot.reset_index(inplace=True)
df_tot.columns = ['year', 'total']

df_tot.head()

In [ ]:
# scatter + linea di regressione adattata + banda di confidenza, in un'unica riga
sns.regplot(x='year', y='total', data=df_tot)
plt.show()

In [ ]:
# personalizzazione: colore, marcatore, dimensione della figura, titolo e assi
plt.figure(figsize=(15, 10))
sns.set(font_scale=1.5)
sns.set_style('whitegrid')

ax = sns.regplot(x='year', y='total', data=df_tot, color='green',
                  marker='+', scatter_kws={'s': 200})

ax.set(xlabel='Year', ylabel='Total Immigration')
ax.set_title('Total Immigration to Canada from 1980 - 2013')
plt.show()

## Punti chiave

- Un **waffle chart** è un'alternativa visiva a un pie chart; con `pywaffle` si crea in un'unica chiamata usando `Waffle` come `FigureClass` di matplotlib.
- Una **nuvola di parole** (`WordCloud` + `.generate()` + `plt.imshow()`) mostra quali parole dominano un testo; `STOPWORDS` filtra parole irrilevanti e `mask` permette di darle una forma personalizzata.
- **Seaborn** (`countplot`, `barplot`, `regplot`) sostituisce molto codice manuale di Matplotlib con funzioni di alto livello pensate per relazioni statistiche.
- `sns.set()` e `sns.set_style()` controllano l'aspetto generale di tutti i grafici della sessione senza toccarli uno per uno.